In [ ]:
import pandas as pd
import pyreadstat
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"C:\Users\Hp\Downloads\Project 2026 DS")# %% 2. Shared definitions
demo_cols=['Age9','Gend3','Eth7','IMD10','Disab3','LondInOut','NSSEC5','Educ6','Orient4','Relig7','ChildAgeU13','Maternity_pop']
group_cols=['LA'] + demo_cols

imd_map={i: f"Decile {i}" for i in range(1,11)}
imd_map[1]="Decile 1 (most deprived)"
imd_map[10]="Decile 10 (least deprived)"

london_boroughs=['Barking and Dagenham','Barnet','Bexley','Brent','Bromley','Camden','City of London','Croydon','Ealing','Enfield','Greenwich','Hackney','Hammersmith and Fulham','Haringey','Harrow','Havering','Hillingdon',
    'Hounslow','Islington','Kensington and Chelsea','Kingston upon Thames','Lambeth','Lewisham','Merton','Newham','Redbridge','Richmond upon Thames''Southwark','Sutton','Tower Hamlets','Waltham Forest','Wandsworth','Westminster']
covid_years=['2020-21','2021-22']

likert_map={'Strongly disagree': 1,'Disagree': 2,'Neither agree nor disagree': 3,'Agree': 4,
    'Strongly agree': 5,'Not asked: done 150 mins of moderate activity in last week': np.nan}
def clean_la_name(name):
    if pd.isna(name):
        return name
    return re.sub(r'^[EWS]\d{8}\s+','',name)

def clean_readiness(series):
    return series.map(likert_map)

def weighted_avg(series,weights):
    d=pd.DataFrame({'v': series,'w': weights}).dropna()
    if d.empty or d['w'].sum() == 0:
        return np.nan
    return np.average(d['v'],weights=d['w'])

In [ ]:
# using readiness as well 
def summarize(x):
    return pd.Series({'pct_active': weighted_avg(x['Filter_Act'],x['wt_final']) * 100,'pct_fairly_active': weighted_avg(x['Filter_InsAct'],
     x['wt_final']) * 100,'pct_inactive': weighted_avg(x['Filter_Inact'],x['wt_final']) * 100,'respondents': len(x),'weighted_base': x['wt_final'].sum()})

def process_year(year_label,filepath,cols_needed,la_col,readyop_col):
    df,meta=pyreadstat.read_sav(filepath,usecols=cols_needed,apply_value_formats=True)

    df[la_col]=df[la_col].astype(str).apply(clean_la_name)
    lon=df[df[la_col].isin(london_boroughs)].copy()
    lon=lon.rename(columns={la_col: 'LA'})

    if pd.api.types.is_categorical_dtype(lon['LA']):
        lon['LA']=lon['LA'].cat.remove_unused_categories()

    print(f"{year_label}: {len(lon)} London respondents, {lon['LA'].nunique()} boroughs")

    for col in group_cols:
        lon[col]=lon[col].astype(object).fillna('Not asked / Not applicable').astype(str)

    imd_raw,_=pyreadstat.read_sav(filepath,usecols=['serial','IMD10'],apply_value_formats=False)
    imd_raw=imd_raw.rename(columns={'IMD10': 'IMD10_raw'})
    lon=lon.merge(imd_raw,on='serial',how='left')
    lon['IMD10_numeric']=lon['IMD10_raw']
    lon['IMD10']=lon['IMD10_raw'].map(imd_map).fillna('Not asked / Not applicable')
    lon=lon.drop(columns=['IMD10_raw'])

    lon[['Filter_Act','Filter_InsAct','Filter_Inact','wt_final']]=lon[['Filter_Act','Filter_InsAct','Filter_Inact','wt_final']].astype(float)

    if 'MEMS7_ALL' in lon.columns:
        lon['MEMS7_ALL']=pd.to_numeric(lon['MEMS7_ALL'],errors='coerce')
    if readyop_col in lon.columns:
        lon['READYOP1_POP']=clean_readiness(lon[readyop_col])
    if 'READYAB1_POP' in lon.columns:
        lon['READYAB1_POP']=clean_readiness(lon['READYAB1_POP'])

    gap=lon.groupby('LA').apply(lambda x: pd.Series({
        'pct_active': weighted_avg(x['Filter_Act'],x['wt_final']) * 100,'pct_fairly_active': weighted_avg(x['Filter_InsAct'],x['wt_final']) * 100,
        'pct_inactive': weighted_avg(x['Filter_Inact'],x['wt_final']) * 100,'mems7_all_wtd': weighted_avg(x['MEMS7_ALL'],x['wt_final']) if 'MEMS7_ALL' in x.columns else np.nan,
        'readiness_ability_wtd': weighted_avg(x['READYAB1_POP'],x['wt_final']) if 'READYAB1_POP' in x.columns else np.nan,'readiness_opportunity_wtd': weighted_avg(x['READYOP1_POP'],x['wt_final']) if 'READYOP1_POP' in x.columns else np.nan,
        'respondents': len(x),'weighted_base': x['wt_final'].sum()})).reset_index()

    gap=gap.rename(columns={'LA': 'borough'})
    gap['survey_year']=year_label
    gap['covid_affected']=year_label in covid_years
    gap['readyop_source_var']=readyop_col 
    gap['source_file']=os.path.basename(filepath)
    rows=[lon.groupby(['LA',col]).apply(summarize).reset_index()
           .rename(columns={'LA': 'borough',col: 'category'}).assign(demographic_group=col)
        for col in demo_cols]
    london_wide_rows=[lon.groupby(col).apply(summarize).reset_index()
           .rename(columns={col: 'category'}).assign(borough='London-wide',demographic_group=col)
        for col in demo_cols]

    profile=pd.concat(rows + london_wide_rows,ignore_index=True)
    profile['survey_year']=year_label
    profile['suppress']=profile['respondents'] < 30
    profile['source_file']=os.path.basename(filepath)
    profile['geography_level']=profile['borough'].apply(lambda b: 'London-wide' if b == 'London-wide' else 'borough')

    return gap,profile

In [ ]:
#all 7 years
all_files={"2016-17": (r"ActiveLives_Data\surveydata1617.sav","LA","READYOP1_POP"),
    "2017-18": (r"ActiveLives_Data\surveydata1718.sav","LA","READYOP1_POP"),
    "2018-19": (r"ActiveLives_Data\surveydata1819.sav","LA_2023","READYOP1_POP"),
    "2019-20": (r"ActiveLives_Data\surveydata1920.sav","LA_2023","READYOP_CV_3_POP"),
    "2020-21": (r"ActiveLives_Data\surveydata2021.sav","LA_2023","READYOP_CV_3_POP"),
    "2021-22": (r"ActiveLives_Data\surveydata2122.sav","LA_2023","READYOP1_POP"),
    "2022-23": (r"ActiveLives_Data\surveydata2223.sav","LA_2023","READYOP1_POP"),}

cols_base=['serial','wt_final','Reg9','LondInOut',
             'Age9','Gend3','Eth7','IMD10','Disab3',
             'NSSEC5','Educ6','Orient4','Relig7',
             'ChildAgeU13','Maternity_pop',
             'Filter_Act','Filter_InsAct','Filter_Inact']

extra_vars=['MEMS7_ALL','READYAB1_POP','READYOP1_POP']

In [ ]:
gap_results=[]
profile_results=[]

for year,(filepath,la_col,readyop_col) in all_files.items():
    _,meta=pyreadstat.read_sav(filepath,metadataonly=True)
    extra_present=[v for v in ['MEMS7_ALL','READYAB1_POP'] if v in meta.column_names]
    if readyop_col in meta.column_names:
        extra_present.append(readyop_col)
    cols_needed=cols_base + [la_col] + extra_present

    gap_y,profile_y=process_year(year,filepath,cols_needed,la_col,readyop_col)
    gap_results.append(gap_y)
    profile_results.append(profile_y)

gapscore_v2=pd.concat(gap_results,ignore_index=True)
gapscore_v2=gapscore_v2[['survey_year','borough','pct_active','pct_fairly_active','pct_inactive','mems7_all_wtd','readiness_ability_wtd','readiness_opportunity_wtd','respondents','weighted_base','covid_affected','readyop_source_var','source_file']]

poplprofile_v2=pd.concat(profile_results,ignore_index=True)
poplprofile_v2=poplprofile_v2[['survey_year','geography_level','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents','weighted_base','suppress','source_file']]

gapscore_v2.to_csv(r"ActiveLives_Data\gapscore_v2.csv",index=False)
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv",index=False)

print(gapscore_v2[['survey_year','readyop_source_var','readiness_opportunity_wtd']].groupby(['survey_year','readyop_source_var']).mean())

In [ ]:
gapscore_v2=pd.concat(gap_results,ignore_index=True)
gapscore_v2=gapscore_v2[['survey_year','borough','pct_active','pct_fairly_active','pct_inactive','mems7_all_wtd','readiness_ability_wtd','readiness_opportunity_wtd','respondents','weighted_base','covid_affected','source_file']]

poplprofile_v2=pd.concat(profile_results,ignore_index=True)
poplprofile_v2=poplprofile_v2[['survey_year','geography_level','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents','weighted_base','suppress','source_file']]

gapscore_v2.to_csv(r"ActiveLives_Data\gapscore_v2.csv",index=False)
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv",index=False)

print("gapscore_v2.csv:",gapscore_v2.shape)
print(gapscore_v2.head())
print("poplprofile_v2.csv:",poplprofile_v2.shape)

In [ ]:

print(gapscore_v2[['survey_year','readiness_ability_wtd','readiness_opportunity_wtd']].groupby('survey_year').mean())

In [ ]:
print(gapscore_v2[gapscore_v2['survey_year'] == '2016-17'][['respondents']].sum())
print(gapscore_v2[gapscore_v2['survey_year'] == '2017-18'][['respondents']].sum())

respondents    19497.0
dtype: float64
respondents    16200.0
dtype: float64


In [ ]:

print("Total rows:",len(poplprofile_v2))

print("rows by geography_level:")
print(poplprofile_v2['geography_level'].value_counts())

print("rows per year:")
print(poplprofile_v2.groupby('survey_year').size())

print("rows per demographic_group:")
print(poplprofile_v2.groupby('demographic_group').size())

print("categories per demographic_group:")
print(poplprofile_v2.groupby('demographic_group')['category'].nunique())

In [ ]:
old_profile=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\poplprofile_merged_ROOT_BACKUP.csv")

print("old file")
print(old_profile['demographic_variable'].value_counts())
print(" new file ")
print(poplprofile_v2['demographic_group'].value_counts())
old_vars=set(old_profile['demographic_variable'].unique())
new_vars=set(poplprofile_v2['demographic_group'].unique())
print("in OLD but not NEW:",old_vars - new_vars)
print("in NEW but not OLD:",new_vars - old_vars)
print("in both:",old_vars & new_vars)

In [ ]:
print(demo_cols)
print(len(demo_cols))

['Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3', 'LondInOut', 'NSSEC5', 'Educ6', 'Orient4', 'Relig7', 'ChildAgeU13', 'Maternity_pop']
12


In [ ]:
old_2018=old_profile[(old_profile['survey_year']=='2018-19') & (old_profile['demographic_variable']=='Age9')]
new_2018=poplprofile_v2[(poplprofile_v2['survey_year']=='2018-19') & (poplprofile_v2['demographic_group']=='Age9')]

print("OLD 2018-19 Age9 total respondents:",old_2018['sample_size'].sum())
print("NEW 2018-19 Age9 total respondents:",new_2018['respondents'].sum())

In [ ]:
#how many not applicable
not_asked=poplprofile_v2[poplprofile_v2['category'] == 'Not asked / Not applicable']
print(f"{len(not_asked)} rows out of {len(poplprofile_v2)} total ({len(not_asked)/len(poplprofile_v2)*100:.1f}%)")
print(not_asked['demographic_group'].value_counts())

In [ ]:
# check were not asked were present the most
not_asked=poplprofile_v2[poplprofile_v2['category'] == 'Not asked / Not applicable']

print("by survey_year:")
print(not_asked['survey_year'].value_counts())
print("by borough:")
print(not_asked['borough'].value_counts().head(10))
print("by geography_level:")
print(not_asked['geography_level'].value_counts())

In [ ]:
not_applicable=poplprofile_v2[poplprofile_v2['category'] == 'Not asked / Not applicable'].copy()
not_applicable.to_csv(r"ActiveLives_Data\poplprofile_v2_notapplicable.csv",index=False)
poplprofile_v2_clean=poplprofile_v2[poplprofile_v2['category'] != 'Not asked / Not applicable'].copy()
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv",index=False)
print("Saved poplprofile_v2_notapplicable.csv:",not_applicable.shape)
print("Saved poplprofile_v2_clean.csv:",poplprofile_v2_clean.shape)
print()
print("Original poplprofile_v2.csv left untouched:",poplprofile_v2.shape)

In [ ]:
old_compare=old_profile.rename(columns={'demographic_variable': 'demographic_group','demographic_category': 'category','sample_size': 'respondents'
})[['survey_year','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents']]

new_compare=poplprofile_v2_clean[['survey_year','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents']]
old_keys=set(zip(old_compare['survey_year'],old_compare['borough'],old_compare['demographic_group'],old_compare['category']))
new_keys=set(zip(new_compare['survey_year'],new_compare['borough'],new_compare['demographic_group'],new_compare['category']))
only_in_old=old_keys - new_keys
only_in_new=new_keys - old_keys

print(f"Rows only in old file: {len(only_in_old)}")
print(f"Rows only in new file: {len(only_in_new)}")
print()
print("rows only in new file")
for k in list(only_in_new)[:15]:
    print(k)

In [ ]:
merged=old_compare.merge(new_compare,on=['survey_year','borough','demographic_group','category'],suffixes=('_old','_new'),how='inner')
merged['pct_inactive_diff']=(merged['pct_inactive_new'] - merged['pct_inactive_old']).abs()
merged['respondents_diff']=merged['respondents_new'] - merged['respondents_old']

print("match rows:",len(merged))
print("distribution of pct_inactive ")
print(merged['pct_inactive_diff'].describe())
print("rows with the most differnec inninactivity")
print(merged.sort_values('pct_inactive_diff',ascending=False)[['survey_year','borough','demographic_group','category','pct_inactive_old','pct_inactive_new','pct_inactive_diff','respondents_old','respondents_new']].head(15))

In [ ]:
extreme_diff_rows=merged[merged['pct_inactive_diff'] > 30]
print("All have respondents < 30:",(extreme_diff_rows['respondents_new'] < 30).all())

In [ ]:
only_in_old_df=old_compare.set_index(['survey_year','borough','demographic_group','category']).loc[list(only_in_old)]
print(only_in_old_df['respondents'].describe())
print(only_in_old_df.sort_values('respondents',ascending=False).head(10))

In [ ]:
old_london_wide=old_compare[old_compare['borough'] == 'London']
new_london_wide=new_compare[new_compare['borough'] == 'London-wide']

print("OLD 'London' rows:",len(old_london_wide))
print("NEW 'London-wide' rows:",len(new_london_wide))

In [ ]:
poplprofile_v2['category']=poplprofile_v2.apply(lambda row: clean_la_name(row['category']) if row['demographic_group'] == 'LondInOut' else row['category'],axis=1)
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv",index=False)
poplprofile_v2_clean=poplprofile_v2[poplprofile_v2['category'] != 'Not asked / Not applicable'].copy()
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv",index=False)

print(poplprofile_v2[poplprofile_v2['demographic_group']=='LondInOut']['category'].unique())

In [ ]:
#ethinicities in bth files
_,meta_old_era=pyreadstat.read_sav(r"ActiveLives_Data\surveydata1718.sav",metadataonly=True)
_,meta_new_era=pyreadstat.read_sav(r"ActiveLives_Data\surveydata2021.sav",metadataonly=True)

print("Eth7 value labels, 2017-18:")
print(meta_old_era.variable_value_labels.get('Eth7'))
print()
print("Eth7 value labels, 2020-21:")
print(meta_new_era.variable_value_labels.get('Eth7'))

In [ ]:
# standardising
poplprofile_v2['category']=poplprofile_v2['category'].replace('Asian (excl. Chinese)','South Asian')
poplprofile_v2.to_csv(r"ActiveLives_Data\poplprofile_v2.csv",index=False)
poplprofile_v2_clean=poplprofile_v2[poplprofile_v2['category'] != 'Not asked / Not applicable'].copy()
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv",index=False)
print(poplprofile_v2[poplprofile_v2['demographic_group']=='Eth7']['category'].unique())

In [ ]:
poplprofile_v2_clean['category']=poplprofile_v2_clean.apply(lambda row: clean_la_name(row['category']) if row['demographic_group'] == 'LondInOut' else row['category'],axis=1)
poplprofile_v2_clean['category']=poplprofile_v2_clean['category'].replace('Asian (excl. Chinese)','South Asian')
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv",index=False)
print("LondInOut categories:",poplprofile_v2_clean[poplprofile_v2_clean['demographic_group']=='LondInOut']['category'].unique())
print("Eth7 categories:",poplprofile_v2_clean[poplprofile_v2_clean['demographic_group']=='Eth7']['category'].unique())
print()
print("Final shape:",poplprofile_v2_clean.shape)

In [ ]:
#check differnec agin between old and new file
old_compare_fixed=old_profile.rename(columns={'demographic_variable': 'demographic_group',
    'demographic_category': 'category','sample_size': 'respondents'
})[['survey_year','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents']]

old_compare_fixed['borough']=old_compare_fixed['borough'].replace('London','London-wide')
new_compare_fixed=poplprofile_v2_clean[['survey_year','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents']]
old_compare_fixed=old_compare_fixed[old_compare_fixed['category'] != 'Not asked / Not applicable']
old_keys=set(zip(old_compare_fixed['survey_year'],old_compare_fixed['borough'],old_compare_fixed['demographic_group'],old_compare_fixed['category']))
new_keys=set(zip(new_compare_fixed['survey_year'],new_compare_fixed['borough'],new_compare_fixed['demographic_group'],new_compare_fixed['category']))
only_in_old=old_keys - new_keys
only_in_new=new_keys - old_keys
print(f"rows only in old {len(only_in_old)}")
print(f"rows only in new {len(only_in_new)}")
print()

if only_in_old:
    print(" only in old")
    for k in sorted(only_in_old)[:10]:
        print(" ",k)

if only_in_new:
    print("only in new:")
    for k in sorted(only_in_new)[:10]:
        print(" ",k)

In [ ]:

poplprofile_v2_clean['category']=poplprofile_v2_clean['category'].replace('NS SEC 9: Students and other','NS SEC 9: Students and other / unclassified')
poplprofile_v2_clean.to_csv(r"ActiveLives_Data\poplprofile_v2_clean.csv",index=False)
print(poplprofile_v2_clean[poplprofile_v2_clean['demographic_group']=='NSSEC5']['category'].unique())

In [ ]:
old_keys=set(zip(old_compare_fixed['survey_year'],old_compare_fixed['borough'],old_compare_fixed['demographic_group'],old_compare_fixed['category']))
new_compare_final=poplprofile_v2_clean[['survey_year','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents']]
new_keys=set(zip(new_compare_final['survey_year'],new_compare_final['borough'],new_compare_final['demographic_group'],new_compare_final['category']))
only_in_old=old_keys - new_keys
print(f"rows only in old {len(only_in_old)}")

In [ ]:

old_keys=set(zip(old_compare_fixed['survey_year'],old_compare_fixed['borough'],old_compare_fixed['demographic_group'],old_compare_fixed['category']))
new_compare_final=poplprofile_v2_clean[['survey_year','borough','demographic_group','category','pct_active','pct_fairly_active','pct_inactive','respondents']]
new_keys=set(zip(new_compare_final['survey_year'],new_compare_final['borough'],new_compare_final['demographic_group'],new_compare_final['category']))

only_in_old=old_keys - new_keys
print(f"Remaining: {len(only_in_old)}")
for k in sorted(only_in_old):
    print(" ",k)

In [ ]:
check=poplprofile_v2_clean[(poplprofile_v2_clean['survey_year'] == '2018-19') &(poplprofile_v2_clean['demographic_group'] == 'NSSEC5')]
print(check['category'].value_counts())
print("Boroughs with 'Aged <16 or 75+' in 2018-19 (new file):",
      check[check['category']=='Aged <16 or 75+']['borough'].nunique())

In [ ]:
df_check,meta_check=pyreadstat.read_sav(r"ActiveLives_Data\surveydata1819.sav",
    usecols=['LA_2023','NSSEC5'],apply_value_formats=True)
df_check['LA_2023']=df_check['LA_2023'].astype(str).apply(clean_la_name)
lon_check=df_check[df_check['LA_2023'].isin(london_boroughs)]
print(lon_check['NSSEC5'].value_counts(dropna=False))

In [ ]:
df_raw,meta_raw=pyreadstat.read_sav(r"ActiveLives_Data\surveydata1819.sav",usecols=['LA_2023','NSSEC5'],
    apply_value_formats=False )
df_raw['LA_2023']=df_raw['LA_2023'].astype(str).apply(clean_la_name)
lon_raw=df_raw[df_raw['LA_2023'].isin(london_boroughs)]
print(lon_raw['NSSEC5'].value_counts(dropna=False))
print("labels in 208-2019")
print(meta_raw.variable_value_labels.get('NSSEC5'))

In [ ]:
#checkimg comsistency
for year,(filepath,la_col,readyop_col) in all_files.items():
    df_check,meta_check=pyreadstat.read_sav(filepath,usecols=['Age9'],apply_value_formats=True)
    print(f"{year} ")
    print(df_check['Age9'].value_counts(dropna=False))

In [ ]:
df_mixed,meta_mixed=pyreadstat.read_sav(r"ActiveLives_Data\surveydata1819.sav",usecols=['LA_2023','NSSEC5'],apply_value_formats=True)
df_raw_only,_=pyreadstat.read_sav( r"ActiveLives_Data\surveydata1819.sav",usecols=['NSSEC5'],apply_value_formats=False)
df_mixed['NSSEC5_raw']=df_raw_only['NSSEC5']

df_mixed['LA_2023']=df_mixed['LA_2023'].astype(str).apply(clean_la_name)
lon_mixed=df_mixed[df_mixed['LA_2023'].isin(london_boroughs)]
print(lon_mixed['NSSEC5_raw'].value_counts(dropna=False))

In [ ]:
all_files_check={"2016-17": r"ActiveLives_Data\surveydata1617.sav",
    "2017-18": r"ActiveLives_Data\surveydata1718.sav","2018-19": r"ActiveLives_Data\surveydata1819.sav","2019-20": r"ActiveLives_Data\surveydata1920.sav","2020-21": r"ActiveLives_Data\surveydata2021.sav","2021-22": r"ActiveLives_Data\surveydata2122.sav","2022-23": r"ActiveLives_Data\surveydata2223.sav",}

os.makedirs(r"ActiveLives_Data\column_lists",exist_ok=True)
for year,path in all_files_check.items():
    _,meta=pyreadstat.read_sav(path,metadataonly=True)
    
    col_df=pd.DataFrame({'column': meta.column_names,'label': meta.column_labels})
    safe_year=year.replace("-","_")
    outpath=fr"ActiveLives_Data\column_lists\columns_{safe_year}.csv"
    col_df.to_csv(outpath,index=False)
    print(f"{year}: saved {len(col_df)} columns to {outpath}")